# POLSCI 3

## Week 14 Section Activity: Natural Experiments, Discontinuities, Differences-in-Differences, and Collective Action

This is an optional activity to let you practice implementing content from Week 13 and 14 in lecture.

In this activity you will practice difference-in-differences designs, natural experiments, regression discontinuity, difference-in-differences, and collective action. The four sections are independent, so they can be done in any order.

If you would like more guidance, there is a second version of this notebook with steps for the questions with multiple components. All of the answers are auto-checked, and you could get the multiple choice questions right by trying every letter, but that will not help you learn the material. Feel free to work with others or ask me for help.

In [ ]:
# RUN THIS CELL
library(testthat)

## Natural Experiments: Charter School Lotteries

In this section, we will think about a natural experiment from lecture: a lottery for admission to a charter school.

-----

**Question 1.** In a study of KIPP Lynn, a charter school, students who applied were admitted by lottery. The researchers want to know the effect of the number of years a student spends at KIPP on math test scores. Which variable is the instrumental variable?

- `'a'`: The number of years a student spends at KIPP
- `'b'`: Whether the student won the admissions lottery
- `'c'`: The student's math test score
- `'d'`: How motivated the student's family is

Put your answer in the quotes below:

In [ ]:
q1.answer <- 'b'

In [ ]:
. = ottr::check("tests/q1.R")

-----

**Question 2.** Which of the following would be a violation of the exclusion restriction in the KIPP lottery study?

- `'a'`: Students who win the lottery are more likely to attend KIPP
- `'b'`: Students who attend KIPP for more years learn more math
- `'c'`: Students who lose the lottery become discouraged, and study less in a way that lowers their test scores no matter which school they attend
- `'d'`: The lottery assigns admission randomly

Put your answer in the quotes below:

In [ ]:
q2.answer <- 'c'

In [ ]:
. = ottr::check("tests/q2.R")

-----

## Regression Discontinuity: Close House Elections

In this section, we will use a regression discontinuity design to study whether electing a Democrat instead of a Republican changes how liberally a representative votes.

The data comes from Lee, Moretti, and Butler (2004) and covers elections to the U.S. House from 1948 to 1990. Each row is one House election in one district, with these variables:

- `state`: name of the state
- `district`: district number
- `year`: year of the election
- `score`: the ADA voting score of the district's representative (higher is more liberal)
- `lagdemvoteshare`: the Democratic share of the vote in the district's previous election (0.5 is a tie)
- `lagdemocrat`: 1 if a Democrat won the previous election, 0 if a Republican did

In [ ]:
# Run this cell first for this section
house <- read.csv("house_elections.csv")
head(house)

-----

**Question 3.** Calculate the average ADA score (`score`) of representatives in close elections, separately for districts where a Democrat just won the previous election and districts where a Democrat just lost it. "Just won" means `lagdemocrat` is 1 and `lagdemvoteshare` is less than 0.52, and "just lost" means `lagdemocrat` is 0 and `lagdemvoteshare` is at least 0.48. Save the average for districts where a Democrat just won in `dem.close.mean` and the average for districts where a Democrat just lost in `rep.close.mean`.

In [ ]:
# Step 1: subset the data to districts where a Democrat just won (lagdemocrat equal to 1 and lagdemvoteshare less than 0.52)
dem.close <- subset(house, lagdemocrat == 1 & lagdemvoteshare < 0.52)

# Step 2: subset the data to districts where a Democrat just lost (lagdemocrat equal to 0 and lagdemvoteshare at least 0.48)
rep.close <- subset(house, lagdemocrat == 0 & lagdemvoteshare >= 0.48)

# Step 3: take the average of score in each subset
dem.close.mean <- mean(dem.close$score)
rep.close.mean <- mean(rep.close$score)

In [ ]:
. = ottr::check("tests/q3.R")

-----

**Question 4.** Which of the following is the best interpretation of the difference between your two averages from Question 3?

- `'a'`: Representatives from districts where a Democrat just won have ADA scores about 21 points higher, on average, than representatives from districts where a Democrat just lost, and this is our best guess of the effect of a Democrat narrowly winning
- `'b'`: Democratic representatives have ADA scores about 21 points higher than Republican representatives in every district
- `'c'`: Winning an election by a narrow margin causes representatives to have ADA scores about 21 percent higher
- `'d'`: The difference only describes districts where a Democrat won by exactly 50 percent of the vote

Put your answer in the quotes below:

In [ ]:
q4.answer <- 'a'

In [ ]:
. = ottr::check("tests/q4.R")

-----

**Question 5.** Why compare districts where a Democrat just won to districts where a Democrat just lost, instead of comparing all districts that elected a Democrat to all districts that elected a Republican?

- `'a'`: Districts that elect Democrats by large margins are different from districts that elect Republicans in many other ways, but whether a Democrat or a Republican wins a very close race is close to random
- `'b'`: Close elections have more voters than other elections
- `'c'`: A close election is the only kind of election where the winner has an effect
- `'d'`: Comparing all districts would not be possible because the data has too many rows

Put your answer in the quotes below:

In [ ]:
q5.answer <- 'a'

In [ ]:
. = ottr::check("tests/q5.R")

-----

## Difference-in-Differences: Organ Donor Registration in California

In this section, we will use a difference-in-differences design to study a change in how California asks about organ donation.

The data comes from Kessler and Roth (2014). In the third quarter of 2011, California changed the question about organ donation that people answer when they get a driver's license, so that they have to actively choose yes or no. The data contains California and 26 other states that can serve as controls, for six quarters. Each row is one state in one quarter, with these variables:

- `state`: name of the state
- `quarter`: the quarter, for example `Q42010` is the fourth quarter of 2010
- `quarter_num`: the quarter as a number, where 1 is the fourth quarter of 2010 and 6 is the first quarter of 2012
- `rate`: the organ donor registration rate in that state and quarter, in percent
- `california`: 1 for California (the treated group), 0 for the other 26 states (the control group)
- `post`: 1 for the quarters after California's change (`quarter_num` 4 to 6), 0 for the quarters before (`quarter_num` 1 to 3)

In [ ]:
# Run this cell first for this section
organ <- read.csv("organ_donations.csv")
head(organ)

# average rate in each quarter, California compared to the control states
quarter.means <- aggregate(rate ~ quarter_num + california, data = organ, FUN = mean)
plot(quarter.means$quarter_num[quarter.means$california == 1], quarter.means$rate[quarter.means$california == 1],
     type = "b", ylim = c(20, 50), xlab = "quarter_num", ylab = "average rate (percent)")
lines(quarter.means$quarter_num[quarter.means$california == 0], quarter.means$rate[quarter.means$california == 0],
      type = "b", lty = 2)
abline(v = 3.5, col = "red")
legend("right", legend = c("California", "Control states"), lty = c(1, 2))

-----

**Question 6.** Calculate the average organ donor registration rate (`rate`) in California (`california` equal to 1) and in the control states (`california` equal to 0), before the policy change (`post` equal to 0) and after it (`post` equal to 1). Save the four averages in `ca.before`, `ca.after`, `other.before`, and `other.after`.

In [ ]:
# Step 1: subset the data to California before and after the change
ca.pre <- subset(organ, california == 1 & post == 0)
ca.post <- subset(organ, california == 1 & post == 1)

# Step 2: subset the data to the control states before and after the change
other.pre <- subset(organ, california == 0 & post == 0)
other.post <- subset(organ, california == 0 & post == 1)

# Step 3: take the average of rate in each subset
ca.before <- mean(ca.pre$rate)
ca.after <- mean(ca.post$rate)
other.before <- mean(other.pre$rate)
other.after <- mean(other.post$rate)

In [ ]:
. = ottr::check("tests/q6.R")

-----

**Question 7.** Using your four averages from Question 6, calculate the difference-in-differences estimate of the effect of California's policy change on the organ donor registration rate. Save it in `did.estimate`.

In [ ]:
did.estimate <- (ca.after - ca.before) - (other.after - other.before)

In [ ]:
. = ottr::check("tests/q7.R")

-----

**Question 8.** Which of the following is the best interpretation of your answer to Question 7?

- `'a'`: After the policy change, California's registration rate was 2.2 percentage points lower than the control states' registration rate
- `'b'`: California's registration rate fell by 2.2 percentage points after the policy change
- `'c'`: If the control states show what would have happened in California without the policy change, the change lowered California's registration rate by about 2.2 percentage points compared to what it would otherwise have been
- `'d'`: The policy change had no effect, because the estimate is close to zero

Put your answer in the quotes below:

In [ ]:
q8.answer <- 'c'

In [ ]:
. = ottr::check("tests/q8.R")

-----

**Question 9.** Which of the following would be a violation of the parallel trends assumption in this study?

- `'a'`: California's registration rate was lower than the control states' registration rate before the policy change
- `'b'`: Around the same time as the policy change, a statewide campaign encouraging Californians to register as organ donors began, and it would have raised California's registration rate even without the policy change
- `'c'`: In the quarter after the policy change, California's registration rate fell and the control states' registration rate rose
- `'d'`: Some of the control states are much smaller than California

Put your answer in the quotes below:

In [ ]:
q9.answer <- 'b'

In [ ]:
. = ottr::check("tests/q9.R")

-----

## Collective Action: The Public Goods Game

In this section, we will read results from the public goods game in Willer (2009).

In the game, groups of six participants each start with $5 and choose how much of it to contribute to a public good. The total contributed is doubled and divided evenly among all six participants, and each participant keeps whatever she did not contribute. After seeing what the others contributed, participants were randomly paired with either a high contributor or a low contributor from their group, and then rated their partner's group motivation (0-100 scale) and played the Fear of Greed game, where the outcome is whether they chose to cooperate (0 or 1).

The estimated effects of being paired with a high contributor instead of a low contributor were:

<table>
<thead>
  <tr><th>Outcome</th><th>Estimate</th><th>Standard Error</th><th>p-value</th></tr>
</thead>
<tbody>
  <tr><td>Participant's cooperation rate in Fear of Greed game (0/1 outcome)</td><td>0.14</td><td>0.11</td><td>0.20</td></tr>
  <tr><td>Rating of partner's group motivation (0-100 scale)</td><td>51.7</td><td>4.0</td><td>0.0001</td></tr>
</tbody>
</table>

-----

**Question 10.** In the public goods game, six participants each start with $5 and choose how much to contribute to the public good. The total contributed is doubled and divided evenly among all six participants, and each participant keeps whatever she did not contribute. Calculate how much money a participant leaves with if she contributes $0 and each of the other five participants contributes $5, and save it in `free.rider.payoff`. Then calculate how much money a participant leaves with if she contributes $5 and each of the other five participants contributes $0, and save it in `contributor.payoff`.

In [ ]:
free.rider.payoff <- 5 + (5 * 5 * 2) / 6
contributor.payoff <- 0 + (5 * 2) / 6

In [ ]:
. = ottr::check("tests/q10.R")

-----

**Question 11.** If everyone contributes $5, each participant leaves with $10, and if no one contributes, each participant leaves with $5. Using your answers to Question 10, which of the following best describes the collective action problem in this game?

- `'a'`: Each participant leaves with more money if she contributes $5, no matter what the others do
- `'b'`: The group as a whole leaves with more money if no one contributes
- `'c'`: Each participant leaves with more money if she contributes $0, no matter what the others do, but every participant leaves with more money if everyone contributes than if no one does
- `'d'`: The doubling of the contributions makes participants better off only if at least three people contribute

Put your answer in the quotes below:

In [ ]:
q11.answer <- 'c'

In [ ]:
. = ottr::check("tests/q11.R")

-----

**Question 12.** The cooperation outcome in the Fear of Greed game is 0 or 1, and the estimate in the table above is 0.14 with a p-value of 0.20. Which of the following is the best interpretation?

- `'a'`: Participants paired with a high contributor were 0.14 percentage points more likely to cooperate on average, and this is statistically significant
- `'b'`: Participants paired with a high contributor were 14 percentage points more likely to cooperate on average, and there is only a 20 percent chance that the effect is real
- `'c'`: 14 percent of participants cooperated, and the rest did not
- `'d'`: Participants paired with a high contributor were 14 percentage points more likely to cooperate on average, but a difference this large could easily have happened by chance even if there were no effect, so we cannot conclude there is no effect

Put your answer in the quotes below:

In [ ]:
q12.answer <- 'd'

In [ ]:
. = ottr::check("tests/q12.R")

-----